In [1]:
import os
import json
import sys
import re
import threading
from concurrent.futures import ThreadPoolExecutor
# Add the project root to the path to allow importing utils
sys.path.append('../../')
import utils.utils as utils
from importlib import reload
reload(utils)

<module 'utils.utils' from '/Users/jlee0/Desktop/research/fine-tuning-or-retrieval/notebooks/FT/../../utils/utils.py'>

In [6]:
# --- 1. Setup ---
PAPER_NAME = "DPO"
PAPER_FILE_PATH = f'../../data/arxiv/cleaned/{PAPER_NAME}.txt'
OUTPUT_DIR = f"../../data/arxiv/explanations/{PAPER_NAME}/"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Reading paper from: {PAPER_FILE_PATH}")
with open(PAPER_FILE_PATH, 'r') as f:
    paper_content = f.read()

# --- 2. Generate Stack Exchange style questions ---
print("Generating student questions about the paper...")
prompt_questions = {
    'system': """### Instructions
You are a confused graduate student reading this research paper. You understand the basics of machine learning but are struggling with specific concepts, derivations, and connections in this paper. Generate a list of at least 20 Stack Exchange style questions that you would ask to clarify your understanding.

Your questions should:
- Be specific and focused on particular aspects of the paper
- Show that you've read the paper but are confused about certain parts
- Ask for intuitive explanations, clarifications of mathematical derivations, or connections to related work
- Be the type of questions that would get good answers on Stack Exchange
- Vary in complexity, from simple to deep.
- Please write any mathematical notation in LaTeX only e.g. "$x^2$" or "$\pi$". Do not use unicode mathematical characters e.g. "π".

For each question, provide:
- A `title` in Stack Exchange question format
- The `question_body` with context and what specifically you're confused about

### Output Format
Provide the output as a JSON object with a single key "questions", which is a list of question dictionaries.
Example:
{
  "questions": [
    {
      "title": "Why does the partition function cancel out in DPO derivation?",
      "question_body": "I'm reading the DPO paper and I understand that they start with the KL-regularized objective, but I'm confused about how the partition function Z(x) cancels out when they move to pairwise preferences. Can someone explain this step intuitively?",
    }
  ]
}""",
    'user': f"### Research Paper\n{paper_content}"
}

response_questions_str = utils.query_llm(
    prompt_questions, 
    model='gpt-5-mini', 
    system_prompt_included=True, 
    return_json=True, 
    max_tokens=10000
)

Reading paper from: ../../data/arxiv/cleaned/DPO.txt
Generating student questions about the paper...


In [ ]:
# Parse the questions response
questions_data = json.loads(response_questions_str)
questions = questions_data['questions']

print(f"Generated {len(questions)} questions")

# --- 3. Generate answers for each question ---
print("Generating answers for questions...")

qa_pairs = []
for i, question in enumerate(questions):
    print(f"Processing question {i+1}/{len(questions)}: {question['title'][:50]}...")
    
    prompt_answer = {
        'system': """### Instructions
You are an expert researcher who specializes in machine learning and optimization. A graduate student has asked a question about a research paper. Provide a clear, detailed Stack Exchange style answer that:

- Directly addresses their confusion
- Provides intuitive explanations alongside technical details
- Uses mathematical notation when helpful but explains it clearly
- Connects to broader concepts when relevant
- Is educational and accessible
- Please write any mathematical notation in LaTeX only e.g. "$x^2$" or "$\pi$". Do not use unicode mathematical characters e.g. "π".

Format your response as a comprehensive Stack Exchange answer.""",
        'user': f"""### Question Title
{question['title']}

### Question Body
{question['question_body']}

### Research Paper Context
{paper_content}"""
    }
    
    answer = utils.query_llm(
        prompt_answer,
        model='gpt-4o-mini',
        system_prompt_included=True,
        max_tokens=2000
    )
    
    qa_pairs.append({
        'title': question['title'],
        'question': question['question_body'],
        'answer': answer
    })

# --- 4. Create single Stack Exchange style explanation file ---
print("Creating Stack Exchange explanation file...")

stackexchange_content = ""
for qa in qa_pairs:
    stackexchange_content += f"### {qa['title']}\n\n**Question:**\n{qa['question']}\n\n**Answer:**\n{qa['answer']}\n\n---\n\n"

# Save all QA pairs in single file
output_file = os.path.join(OUTPUT_DIR, "stackexchange.txt")
with open(output_file, 'w') as f:
    f.write(stackexchange_content)

print(f"Saved all Q&A pairs to {output_file}")


Generated 24 questions
Generating answers for questions...
Processing question 1/24: Derivation of the optimal policy form in Eq. (6): ...
Processing question 2/24: Why and how does the partition function cancel whe...
Processing question 3/24: Intuition for the DPO reparameterization r(x,y)=\b...
